<a href="https://colab.research.google.com/github/Arjunanand717/EXP-10-/blob/main/EXP_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PART A: ENVIRONMENT SETUP AND MODEL SELECTION

In this section, we install the necessary Python packages (`transformers`, `torch`, `gradio`) and load a lightweight instruction-following model suitable for Google Colab CPU/GPU runtimes.

In [ ]:
# Student Roll No: [24BAD401]
# Install Hugging Face Transformers, PyTorch, and Gradio
!pip install -q transformers torch gradio

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Using a lightweight, high-performance instruction model (SmolLM2-135M-Instruct)
model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
print(f"Model loaded successfully on {device}!")

Loading tokenizer...


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded successfully on cpu!


## PART B: PROMPT HANDLING AND RESPONSE GENERATION

Here we define a robust function that handles user input, formats the prompt using the model's recommended chat template, generates a response, and includes error handling.

In [ ]:
def generate_response(prompt: str, max_new_tokens: int = 150, temperature: float = 0.7) -> str:
    # Handle empty input
    if not prompt or not prompt.strip():
        return "Error: Input prompt cannot be empty."

    try:
        # Format prompt using the model's chat template
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # Tokenize inputs
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode only the newly generated tokens
        input_len = inputs["input_ids"].shape[1]
        response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
        return response.strip()

    except Exception as e:
        return f"An error occurred during generation: {str(e)}"

### Testing with Five Different Prompts

In [ ]:
test_prompts = [
    "What is the capital of France?",
    "Explain gravity in one simple sentence.",
    "Write a short, professional email subject line for scheduling a meeting.",
    "Give me a quick recipe for a healthy breakfast.",
    "What is 15 multiplied by 4?"
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"--- Test Prompt {i} ---")
    print(f"Prompt: {prompt}")
    response = generate_response(prompt)
    print(f"Response: {response}\n")

--- Test Prompt 1 ---
Prompt: What is the capital of France?
Response: The capital of France is Paris. It is a city known for its historical landmarks, cultural institutions, and culinary delights. Paris is a vibrant city with a rich history, art, and music scene, making it a destination for tourists and expats alike.

--- Test Prompt 2 ---
Prompt: Explain gravity in one simple sentence.
Response: Gravity is a force that attracts objects with mass towards each other, pulling them towards the center of the Earth and causing them to fall towards the ground.

--- Test Prompt 3 ---
Prompt: Write a short, professional email subject line for scheduling a meeting.
Response: "Confirmation of Meeting Schedule:

Subject: Confirming Meeting Schedule

Dear Team,

We hope this email finds you well. We have scheduled a meeting for [date] with the [team_name], where we will discuss [topic]. Please ensure you arrive on time to avoid any delays.

Best regards,
[Your Name]

[Your Position]
[Your Positio

## PART C & D: BUILDING THE CHATBOT INTERFACE AND DEPLOYMENT

We will use Gradio to build a clean chat interface. Gradio natively handles conversation history and provides a shareable public link when `share=True` is set.

In [ ]:
import gradio as gr

def chatbot_response(message, history):
    # Reconstruct the conversation context from the Gradio history list safely
    messages = []
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    # Add the current user message
    messages.append({"role": "user", "content": message})

    try:
        # Format dialogue context using chat template
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )

        input_len = inputs["input_ids"].shape[1]
        response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
        return response.strip()
    except Exception as e:
        return f"Error: {str(e)}"

# Create a Gradio ChatInterface without 'type'
demo = gr.ChatInterface(
    fn=chatbot_response,
    title="AI Assistant Chatbot",
    description="A lightweight Transformer-based chatbot running directly on Hugging Face models in Colab.",
    examples=["Hello!", "How can I plan my day effectively?", "Tell me a funny joke."]
)

# Launching with share=True creates a public temporary deployment link
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b798a31a497b2f212e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
